# Kiểm tra dữ liệu đầu vào — Issue #2

## 1. Mục tiêu và nguồn dữ liệu

Dataset validation kiểm tra xem tệp đầu vào có đúng cấu trúc và đáp ứng các quy tắc chất lượng đã định trước khi dùng cho EDA hoặc huấn luyện. Bước này giúp phát hiện sớm tệp sai, thiếu cột hoặc giá trị khó xử lý ở các bước sau.

Bộ [Kaggle Playground Series S4E2](https://www.kaggle.com/competitions/playground-series-s4e2/data) gồm `train.csv` có nhãn dùng trong phần này, `test.csv` không có nhãn dùng để dự đoán khi dự thi, và `sample_submission.csv` minh họa định dạng nộp bài.

## 2. Schema và bảy nhãn

Số dòng giúp phát hiện tệp thiếu hoặc sai phiên bản. Tên cột, số cột và kiểu giá trị bảo đảm mỗi trường được diễn giải đúng. Schema dự kiến có 18 cột, gồm `id` là số nguyên, 16 đặc trưng số hoặc category, và `NObeyesdad` là nhãn đích. `id` chỉ dùng để định danh bản ghi.

Bảy nhãn biểu thị các nhóm cân nặng trong bộ dữ liệu:

- `Insufficient_Weight` — thiếu cân.
- `Normal_Weight` — cân nặng bình thường.
- `Overweight_Level_I` và `Overweight_Level_II` — thừa cân mức I và II.
- `Obesity_Type_I`, `Obesity_Type_II` và `Obesity_Type_III` — béo phì loại I, II và III.

## 3. Kiểm tra chất lượng

- **Missing values:** đếm ô trống và các dấu hiệu như `NA` hoặc `NULL` theo từng cột. Cột không có mặt được báo là lỗi schema.
- **Duplicate:** ID trùng là ID hợp lệ lặp lại. Trùng toàn dòng so khớp tất cả các trường. Trùng khi bỏ ID so khớp nội dung còn lại, kể cả nhãn. Mỗi phép đếm ghi số lần lặp sau lần xuất hiện đầu tiên.
- **Bất thường:** cột số được kiểm tra khả năng đọc thành số hữu hạn và so với khoảng sàng lọc. Cột category được so khớp chính xác với danh sách cho phép.

## 4. Cách chạy và đọc báo cáo

Dùng kernel Python 3.11 của dự án. Từ thư mục gốc, chạy `python -m src.data_loader` để xuất hai bảng vào `outputs/tables/`. Mã thoát 0 nghĩa là mọi phép kiểm tra đạt, 1 nghĩa là đã xuất báo cáo có mục được đánh dấu, và 2 nghĩa là lệnh không hoàn tất.

- **`dataset_validation_checks.csv`:** `check` là tên phép kiểm tra, `column` là cột liên quan (trống nếu áp dụng cho cả bảng), `count` là số quan sát hoặc số vi phạm tùy phép kiểm tra, `criterion` là quy tắc, và `passed` cho biết quy tắc có đạt hay không. Mục không xác định có thể để trống số đếm và trạng thái. Ví dụ, `row_count` có `count` là số dòng quan sát, còn `missing` có `count` là số ô thiếu.
- **`dataset_validation_labels.csv`:** `label` là nhãn, `status` phân biệt nhãn dự kiến, nhãn lạ, nhãn thiếu hoặc cột nhãn không có mặt, và `count` là số dòng tương ứng. Khi schema hợp lệ, đối chiếu tổng của bảy nhãn dự kiến với tổng số dòng train.

## 5. Diễn giải kết quả đã lưu

Tệp local có **20.758 dòng, 18 cột và 16 đặc trưng**. Cả bảy nhãn đều xuất hiện và tổng số đếm bằng 20.758. Theo các quy tắc hiện tại, số ô thiếu, số lần trùng ở cả ba phép kiểm tra và số giá trị bị đánh dấu bất thường đều bằng 0.

Các khoảng số chỉ là quy tắc sàng lọc. Schema hợp lệ vẫn có thể đi kèm nhãn sai, thiên lệch hoặc các bản ghi gần giống nhau. Phép trùng khi bỏ ID vẫn giữ nhãn, nên các dòng cùng đặc trưng nhưng khác nhãn cần được xem xét riêng trước khi chia tập.


In [10]:
from hashlib import sha256
from pathlib import Path
import sys

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
if not (root / "src" / "data_loader.py").is_file():
    raise RuntimeError("Open this notebook from the repository root or notebooks directory.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.data_loader import read_csv, validate_dataset

source = root / "data" / "train.csv"
if not source.is_file():
    print("Pending: download the official Kaggle archive and place train.csv in data/.")
else:
    before = sha256(source.read_bytes()).hexdigest()
    columns, rows = read_csv(source)
    checks, labels = validate_dataset(columns, rows)
    after = sha256(source.read_bytes()).hexdigest()
    if before != after:
        raise RuntimeError("The input CSV changed during validation.")

    print(f"Input SHA-256: {after}")
    print("Columns:", ", ".join(columns))
    print("Check | Column | Count | Criterion | Passed")
    for check, column, count, criterion, passed in checks:
        value = "unavailable" if count is None else count
        print(f"{check} | {column} | {value} | {criterion} | {passed}")
    print("Label | Status | Count")
    for label, status, count in labels:
        print(f"{label} | {status} | {count}")


Input SHA-256: fec835cc642ffbc3274a026abfe465d0aaceedfe2f7b35d987029f676d7dae6d
Columns: id, Gender, Age, Height, Weight, family_history_with_overweight, FAVC, FCVC, NCP, CAEC, SMOKE, CH2O, SCC, FAF, TUE, CALC, MTRANS, NObeyesdad
Check | Column | Count | Criterion | Passed
row_count |  | 20758 | Expected 20758 | True
column_count |  | 18 | Expected 18 | True
feature_count |  | 16 | Expected 16 | True
missing | id | 0 | Blank, NA, N/A, NaN or NULL | True
invalid_id | id | 0 | Nonnegative decimal integer, at most 18 digits | True
duplicate_id | id | 0 | Repeated valid integer ID, excluding first occurrence | True
missing | Gender | 0 | Blank, NA, N/A, NaN or NULL | True
unexpected_category | Gender | 0 | Exact match to Female | Male | True
missing | Age | 0 | Blank, NA, N/A, NaN or NULL | True
invalid_numeric | Age | 0 | Finite number required | True
out_of_range | Age | 0 | Inclusive screening range [0, 120] | True
missing | Height | 0 | Blank, NA, N/A, NaN or NULL | True
invalid_numeri


# Phân tích khám phá dữ liệu (EDA) — Issue #3

## Mục tiêu

Phân tích dữ liệu trước khi xây dựng mô hình Machine Learning.

Chúng ta sẽ:
- Xem số lượng và tỷ lệ của 7 lớp cân nặng.
- Thống kê các đặc trưng Age, Height và Weight.
- Quan sát sự khác biệt giữa các lớp.

Các kết quả thống kê được lấy từ module `src.eda`
đã được xây dựng và kiểm thử trong Issue #3.


In [ ]:

# Sử dụng lại code thống kê của Issue #3.
import pandas as pd
from IPython.display import display

from src.data_loader import read_csv
from src.eda import build_tables

# Đọc dữ liệu từ train.csv.
eda_columns, eda_rows = read_csv(source)

# Tính các bảng thống kê.
eda_tables = build_tables(eda_columns, eda_rows)

# Hiển thị phân bố bảy lớp cân nặng.
label_df = pd.DataFrame(eda_tables["label_distribution"])

print("PHÂN BỐ BẢY LỚP CÂN NẶNG")
display(label_df[["label", "count", "proportion"]])

# Hiển thị thống kê các đặc trưng số.
numeric_df = pd.DataFrame(eda_tables["numeric_summary"])

selected_features = numeric_df[
    numeric_df["feature"].isin(["Age", "Height", "Weight"])
]

print("THỐNG KÊ AGE, HEIGHT VÀ WEIGHT")
display(
    selected_features[
        ["feature", "mean", "std", "min", "median", "max"]
    ]
)



# Trực quan hóa dữ liệu — Issue #4

## 1. Phân bố bảy lớp cân nặng

**Mục tiêu:** Quan sát số lượng mẫu thuộc từng lớp
`NObeyesdad` và đánh giá mức độ cân bằng nhãn.

**Phương pháp:** Biểu đồ cột ngang (Horizontal Bar Plot).

Biểu đồ được tạo bằng hàm `plot_label_distribution()`
trong module `src.visualization`.


In [ ]:

# Import hàm vẽ biểu đồ đã viết trong Issue #4.
from IPython.display import Image, display
from src.visualization import plot_label_distribution

# Đọc dữ liệu vào DataFrame của Pandas.
train_df = pd.read_csv(source)

# Tạo và lưu biểu đồ số 1.
plot_label_distribution(train_df)

# Hiển thị hình PNG ngay trong notebook.
image_path = root / "outputs" / "figures" / "01_label_distribution.png"

display(Image(filename=str(image_path)))



### Nhận xét

Biểu đồ cho thấy cả bảy lớp cân nặng đều có mặt
trong bộ dữ liệu.

- `Obesity_Type_III` có nhiều mẫu nhất: 4.046 mẫu
  (khoảng 19,49%).
- `Overweight_Level_I` có ít mẫu nhất: 2.427 mẫu
  (khoảng 11,69%).
- Số lượng mẫu giữa các lớp có sự chênh lệch.

**Ý nghĩa đối với Machine Learning:**

Khi huấn luyện mô hình phân loại, cần xem xét phân bố
các lớp để tránh đánh giá mô hình chỉ dựa trên Accuracy.
Có thể sử dụng thêm Precision, Recall, F1-score và
Confusion Matrix trong giai đoạn đánh giá mô hình.



## 2. Phân bố độ tuổi (Age)

**Mục tiêu:** Khám phá khoảng tuổi và mức độ tập trung
của các mẫu dữ liệu.

**Phương pháp:** Histogram kết hợp đường KDE.

Histogram cho biết số lượng mẫu trong từng khoảng tuổi,
còn KDE giúp quan sát hình dạng phân bố tổng thể.


In [ ]:

# Tạo biểu đồ phân bố độ tuổi.
from src.visualization import plot_age_distribution

plot_age_distribution(train_df)

# Hiển thị hình vừa được tạo.
age_image_path = root / "outputs" / "figures" / "02_age_distribution.png"
display(Image(filename=str(age_image_path)))



### Nhận xét

Độ tuổi trong dữ liệu dao động từ 14 đến 61 tuổi,
trung bình khoảng 23,84 tuổi và trung vị khoảng 22,82 tuổi.

Phần lớn mẫu tập trung ở nhóm người trẻ, đặc biệt
trong khoảng 18–27 tuổi.

Phân bố có đuôi kéo dài về phía các độ tuổi cao hơn,
cho thấy dữ liệu không phân bố đồng đều giữa các nhóm tuổi.

**Ý nghĩa đối với ML:** Cần lưu ý phân bố tuổi khi
đánh giá khả năng khái quát của mô hình sang các nhóm
tuổi ít xuất hiện trong dữ liệu.



## 3. Phân bố chiều cao (Height)

**Mục tiêu:** Quan sát phạm vi và mức độ phân tán
của chiều cao trong bộ dữ liệu.

**Phương pháp:** Histogram kết hợp KDE.

Đơn vị của chiều cao là mét (m).


In [ ]:

# Tạo biểu đồ phân bố chiều cao.
from src.visualization import plot_height_distribution

plot_height_distribution(train_df)

# Hiển thị biểu đồ.
height_image_path = root / "outputs" / "figures" / "03_height_distribution.png"
display(Image(filename=str(height_image_path)))



### Nhận xét

Chiều cao dao động từ khoảng 1,45 m đến 1,98 m.

Chiều cao trung bình khoảng 1,7002 m, trung vị khoảng
1,70 m và độ lệch chuẩn mẫu khoảng 0,0873 m.

Phần lớn mẫu tập trung trong khoảng 1,60–1,80 m.
Phân bố có nhiều đỉnh quan sát được.

**Ý nghĩa đối với ML:** Chiều cao có thể kết hợp
với cân nặng để cung cấp thêm thông tin cho bài toán
phân loại mức độ cân nặng.



## 4. Phân bố cân nặng (Weight)

**Mục tiêu:** Quan sát khoảng giá trị, mức độ tập trung
và hình dạng phân bố cân nặng trong dữ liệu.

**Phương pháp:** Histogram kết hợp KDE.

Đơn vị cân nặng là kilogram (kg).


In [ ]:

# Tạo và hiển thị biểu đồ phân bố cân nặng.
from src.visualization import plot_weight_distribution

plot_weight_distribution(train_df)

weight_image_path = (
    root / "outputs" / "figures" / "04_weight_distribution.png"
)

display(Image(filename=str(weight_image_path)))



### Nhận xét

Cân nặng dao động từ khoảng 39 đến 165,06 kg.

Giá trị trung bình khoảng 87,89 kg, trung vị khoảng
84,06 kg và độ lệch chuẩn mẫu khoảng 26,38 kg.

Phân bố có nhiều vùng tập trung khác nhau, thay vì
chỉ có một đỉnh rõ ràng.

**Ý nghĩa đối với ML:** Weight là đặc trưng cần xem xét
trong bài toán phân loại mức độ cân nặng. Tuy nhiên,
không nên sử dụng riêng Weight để giải thích toàn bộ nhãn,
vì Height và các đặc trưng khác cũng có thể liên quan.



## 5. Phân bố các đặc trưng phân loại

**Mục tiêu:** Quan sát số lượng mẫu thuộc từng nhóm
của các đặc trưng phân loại.

Các đặc trưng được xem xét:
- Gender: giới tính.
- family_history_with_overweight: tiền sử gia đình thừa cân.
- FAVC: thường xuyên ăn thực phẩm nhiều calo.

**Phương pháp:** Countplot.


In [ ]:

# Tạo và hiển thị ba Countplot trong cùng một hình.
from src.visualization import plot_categorical_distributions

plot_categorical_distributions(train_df)

categorical_image_path = (
    root / "outputs" / "figures" / "05_categorical_distributions.png"
)

display(Image(filename=str(categorical_image_path)))



### Nhận xét

Phân bố giới tính tương đối cân bằng giữa Male và Female.

Đối với family_history_with_overweight, nhóm `yes`
chiếm đa số.

Tương tự, với FAVC, số mẫu thuộc nhóm `yes` lớn hơn
đáng kể so với nhóm `no`.

**Ý nghĩa đối với ML:** Các đặc trưng phân loại có thể
cung cấp thông tin hữu ích cho mô hình. Tuy nhiên,
sự chênh lệch số lượng giữa hai nhóm của một đặc trưng
không tự động chứng minh đặc trưng đó gây ra béo phì
hoặc có khả năng dự đoán tốt.



## 6. Phân bố cân nặng theo bảy lớp

**Mục tiêu:** So sánh phân bố cân nặng giữa các lớp
của biến mục tiêu NObeyesdad.

**Phương pháp:** Boxplot.

Biểu đồ thể hiện trung vị, khoảng tứ phân vị,
râu hộp và những điểm nằm ngoài râu.


In [ ]:

# Tạo và hiển thị Boxplot theo bảy nhãn.
from src.visualization import plot_weight_by_label

plot_weight_by_label(train_df)

boxplot_image_path = (
    root / "outputs" / "figures" / "06_weight_by_label_boxplot.png"
)

display(Image(filename=str(boxplot_image_path)))



### Nhận xét

Phân bố cân nặng khác nhau giữa bảy lớp NObeyesdad.

Các nhóm có khoảng cân nặng chồng lấn nhau,
cho thấy một giá trị cân nặng có thể xuất hiện
ở nhiều lớp khác nhau.

Đáng chú ý, trung vị cân nặng của Obesity_Type_III
thấp hơn Obesity_Type_II trong bộ dữ liệu này.

Biểu đồ cũng xuất hiện các điểm nằm ngoài râu hộp.
Những điểm này không nhất thiết là dữ liệu sai.

**Ý nghĩa đối với ML:** Cần kết hợp thêm Height và
những đặc trưng khác để phân biệt các mức độ cân nặng,
thay vì chỉ dựa vào Weight.



## 7. Quan hệ giữa chiều cao và cân nặng theo nhãn

**Mục tiêu:** Khám phá mối quan hệ giữa Height,
Weight và bảy lớp NObeyesdad.

**Phương pháp:** Scatter Plot.

Mỗi điểm biểu diễn một bản ghi dữ liệu:
- Trục X: chiều cao (m).
- Trục Y: cân nặng (kg).
- Màu sắc: lớp NObeyesdad.


In [ ]:

# Tạo và hiển thị Scatter Plot theo bảy lớp.
from src.visualization import plot_height_weight_scatter

plot_height_weight_scatter(train_df)

scatter_image_path = (
    root / "outputs" / "figures" / "07_height_weight_scatter.png"
)

display(Image(filename=str(scatter_image_path)))



### Nhận xét

Các lớp mức độ cân nặng có xu hướng xuất hiện
ở những vùng khác nhau trên mặt phẳng Height–Weight.

Nhóm Insufficient_Weight chủ yếu tập trung ở vùng
cân nặng thấp. Các nhóm béo phì thường xuất hiện
ở vùng cân nặng cao hơn.

Tuy nhiên, nhiều vùng có các lớp chồng lấn nhau,
cho thấy không thể phân biệt hoàn toàn bảy lớp
chỉ bằng hai đặc trưng Height và Weight.

**Ý nghĩa đối với ML:** Việc kết hợp nhiều đặc trưng
có thể giúp mô hình phân loại tốt hơn. Cần kiểm tra
điều này bằng kết quả huấn luyện và đánh giá thực tế.
